# BTP — Two Cells in a Closed Box, Walking, Colliding, Merging

## The scenario

Two cells, each running its own independent decide → deform → move →
relax → decide-again cycle from `BTP_version3_run_and_tumble.ipynb`,
inside a box with walls instead of the periodic domain every previous
notebook used. When their paths cross, **they merge into one cell**
rather than repelling (the opposite choice from the Version 4 two-cell
collision experiment, where a steric term was added specifically to
*prevent* merging). At the moment of contact, the merged cell picks a
new direction, informed by the two cells' momentum going into the
collision.

## Three design decisions, and why

**1. Merging is free — use one shared field, not two.**
The Version 4 two-cell work kept two *separate* phase fields,
$\phi_1,\phi_2$, specifically so the cells would stay distinguishable
and a steric term was needed to stop them overlapping. Here we want
the opposite: once two blobs of the same $\phi$ touch, ordinary
interfacial tension — the same physics already validated in Version
3's passive-limit test — makes them coalesce into one connected region
with no extra code. So this notebook uses **one** $\phi$ field and
**one** $\mathbf P$ field, seeded with **two disjoint blobs** at $t=0$.
Nothing distinguishes "cell 1 material" from "cell 2 material" once
they touch, which is exactly what "mix into one another" means for a
diffuse-interface model: there is no boundary left to separate them.

**2. A closed box, implemented as a soft wall rather than a new solver.**
The Stokes solve in every version so far is spectral (`np.fft.fft2`),
which is only valid on a periodic domain — there is no "wall" boundary
condition built into that method. Properly replacing it with a
non-periodic solver (finite differences/volumes with a real no-slip
or free-slip wall, needing a sparse linear solve instead of two FFTs)
is a substantial rewrite. The pragmatic middle ground, and the one
used here: keep the periodic spectral solve exactly as is, but add
**one new term** to $\mu_\phi$ — a soft potential that ramps up sharply
near the domain edges and pushes $\phi$ back inward. This is the same
kind of trick as the Version 4 steric-repulsion term, just against a
static wall field instead of another cell's $\phi$. It is not a
literal rigid wall with proper hydrodynamics at the boundary, and the
strength/width of the potential below ($k_{\rm wall}$, the margin) are
reasoned starting guesses, not tuned by execution — you should watch
the first run and strengthen it if a cell ever reaches the literal
edge of the array. A true non-periodic Stokes solve is listed in
Section 7 as the honest way to remove this approximation later.

**3. "Conservation of angular momentum" doesn't quite apply here —
so here is what we do instead.**
This is worth being direct about rather than quietly working around.
Stokes flow is the *zero-Reynolds-number* limit: inertia is dropped
from the equations entirely (Version 3's own justification was
$Re\sim10^{-5}$). Momentum is not carried forward in time — the force
and torque on the fluid balance **instantaneously** at every step,
dissipated immediately by viscosity and substrate friction. A
swimmer in this regime famously has no coasting: stop paddling and it
stops *now*, not gradually (Purcell, "Life at Low Reynolds Number").
So there is no literal dynamical quantity called "the system's angular
momentum" that persists across a collision for us to conserve.

What we do instead, which captures the spirit of the suggestion
without claiming a conservation law the equations don't have: just
before the two blobs touch, we compute the **linear and angular
momentum each cell carries from its own crawling motion** —
$\mathbf p_i = m_i\mathbf v_i$ and $L_i=m_i(\mathbf r_i-\mathbf
R_{\rm cm})\times\mathbf v_i$, with $m_i=\int\phi_i\,dA$ the cell's
mass-like weight — and use the combined $\mathbf p_{\rm tot}$ as a
**bias**, not a constraint, on the merged cell's next randomly chosen
direction: instead of a uniform random heading, we draw from a von
Mises distribution centred on the direction of $\mathbf p_{\rm tot}$.
$L_{\rm tot}$ is reported as a diagnostic alongside it. This treats
the pre-collision state as *information fed into a stochastic
decision*, which is a legitimate modelling choice, rather than as a
dynamical quantity the (inertia-free) equations themselves conserve.


# 1. The Version 3 physics, with one new term

Grid, derivatives, molecular field, body force, Stokes solve, diagnostics and drawing are copied from `BTP_version3.ipynb` unchanged. `chemical_potential_phi` gets exactly one new line: the soft wall potential described above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Video
from scipy import ndimage


# ---------------- grid (bigger than the single-cell notebooks: room for ----
# ---------------- two cells, a box margin, and a walk between them)    ----
N = 192
L = 120.0

# ---------------- cell ----------------
R0 = 13.0
eps = 1.5
a_phi = 1.0
M_phi = 1.0

# ---------------- polarization ----------------
a_pol = 0.5
kappa = 1.0
Gamma = 1.0
chi = 1.1

# ---------------- activity (used while a cell is in its "move" phase) ----
w0 = 1.0
zeta = -1.0
beta = 0.25

# ---------------- substrate and fluid ----------------
eta = 1.0
xi = 1.0

# ---------------- the new bit: a soft confining wall ----------------
wall_margin_frac = 0.80   # the potential turns on beyond this fraction of L/2
wall_sharpness = 4.0      # higher = a steeper, more wall-like rise
k_wall = 40.0             # strength -- reasoned, NOT tuned by execution; watch
                          # the first run and raise this if phi ever reaches
                          # the literal edge of the array

# ---------------- time ----------------
dt = 0.01
sample_every = 100

print(f"grid {N} x {N},  dx = {L/N:.4f}")
print(f"stability: dt must be < dx^2/(4 M eps^2) = "
      f"{(L/N)**2/(4*M_phi*eps**2):.4f};  using dt = {dt}")


In [ ]:
def setup_grid(N_=N, L_=L):
    global N, L, dx, X, Y, KX, KY, K2, K2_safe
    N, L = N_, L_
    dx = L / N
    coords = (np.arange(N) - N // 2) * dx
    X, Y = np.meshgrid(coords, coords, indexing="ij")
    k1 = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    KX, KY = np.meshgrid(k1, k1, indexing="ij")
    K2 = KX**2 + KY**2
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0


setup_grid()


def d_dx(a):
    return (np.roll(a, -1, axis=0) - np.roll(a, 1, axis=0)) / (2.0 * dx)


def d_dy(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)


def laplacian(a):
    return (np.roll(a, -1, axis=0) + np.roll(a, 1, axis=0)
            + np.roll(a, -1, axis=1) + np.roll(a, 1, axis=1)
            - 4.0 * a) / dx**2


def divergence_of_flux(phi, ux, uy):
    Fx = 0.5 * (phi + np.roll(phi, -1, axis=0)) * 0.5 * (ux + np.roll(ux, -1, axis=0))
    Fy = 0.5 * (phi + np.roll(phi, -1, axis=1)) * 0.5 * (uy + np.roll(uy, -1, axis=1))
    return ((Fx - np.roll(Fx, 1, axis=0)) + (Fy - np.roll(Fy, 1, axis=1))) / dx


In [ ]:
def make_wall_field(margin_frac=wall_margin_frac, sharpness=wall_sharpness):
    '''0 in the interior; rises smoothly to 1 right at the box edge.
    A square (not circular) confining region, matching the square domain.'''
    half = L / 2.0
    wall_start = margin_frac * half
    dx_edge = np.maximum(0.0, np.abs(X) - wall_start)
    dy_edge = np.maximum(0.0, np.abs(Y) - wall_start)
    d = np.sqrt(dx_edge**2 + dy_edge**2)
    span = max(half - wall_start, 1e-9)
    return np.clip(d / span, 0.0, 1.0) ** sharpness


WALL = make_wall_field()
print(f"wall zone starts at |x| or |y| > {wall_margin_frac * L/2:.1f} "
      f"(box half-width is {L/2:.1f})")


In [ ]:
def chemical_potential_phi(phi, P2, dPx_dx, dPy_dy):
    return (2.0 * a_phi * phi * (phi - 1.0) * (2.0 * phi - 1.0)
            - eps**2 * laplacian(phi)
            - a_pol * P2
            - beta_run * (dPx_dx + dPy_dy)
            + k_wall * WALL * phi)                      # <-- new: soft wall


def molecular_field_P(phi, Px, Py, P2):
    bulk = -a_pol * (2.0 * phi - 1.0) + a_pol * P2
    hx = bulk * Px - kappa_run * laplacian(Px) + beta_run * d_dx(phi)
    hy = bulk * Py - kappa_run * laplacian(Py) + beta_run * d_dy(phi)
    return hx, hy


In [ ]:
def body_force(phi, Px, Py, mu, hx, hy):
    Axx = -zeta_run * phi * Px * Px
    Axy = -zeta_run * phi * Px * Py
    Ayy = -zeta_run * phi * Py * Py

    torque = 0.5 * (Px * hy - Py * hx)
    Exx = -chi_run * Px * hx
    Exy = torque - 0.5 * chi_run * (Px * hy + Py * hx)
    Eyx = -torque - 0.5 * chi_run * (Py * hx + Px * hy)
    Eyy = -chi_run * Py * hy

    Pxx, Pxy = d_dx(Px), d_dy(Px)
    Pyx, Pyy = d_dx(Py), d_dy(Py)
    Kxx = -kappa_run * (Pxx * Pxx + Pyx * Pyx)
    Kxy = -kappa_run * (Pxx * Pxy + Pyx * Pyy)
    Kyy = -kappa_run * (Pxy * Pxy + Pyy * Pyy)

    Sxx = Axx + Exx + Kxx
    Sxy = Axy + Exy + Kxy
    Syx = Axy + Eyx + Kxy
    Syy = Ayy + Eyy + Kyy

    fx = d_dx(Sxx) + d_dy(Sxy) - phi * d_dx(mu)
    fy = d_dx(Syx) + d_dy(Syy) - phi * d_dy(mu)
    return fx, fy


def stokes_solve(fx, fy):
    fx_h = np.fft.fft2(fx)
    fy_h = np.fft.fft2(fy)
    longitudinal = (KX * fx_h + KY * fy_h) / K2_safe
    fx_h = fx_h - KX * longitudinal
    fy_h = fy_h - KY * longitudinal
    screened = xi + eta * K2
    vx_h = fx_h / screened
    vy_h = fy_h / screened
    vx_h[0, 0] = 0.0
    vy_h[0, 0] = 0.0
    return np.real(np.fft.ifft2(vx_h)), np.real(np.fft.ifft2(vy_h))


In [ ]:
def centre_of_mass(phi):
    mass = phi.sum()
    return (X * phi).sum() / mass, (Y * phi).sum() / mass


def touches_boundary(phi, margin=6):
    '''Should never fire if k_wall is strong enough -- a safety check,
    not the confinement mechanism itself.'''
    inside = phi > 0.5
    return bool(inside[:margin].any() or inside[-margin:].any()
                or inside[:, :margin].any() or inside[:, -margin:].any())


def calculate_diagnostics(phi, Px, Py, vx, vy):
    inside = phi > 0.5
    area = inside.sum() * dx**2
    gx, gy = d_dx(phi), d_dy(phi)
    perimeter = np.sqrt(gx**2 + gy**2).sum() * dx**2
    circularity = 4.0 * np.pi * area / max(perimeter**2, 1e-12)
    return {"area": area, "circularity": circularity, "v_max": np.sqrt(vx**2 + vy**2).max()}


def draw_cell(ax, phi, Ax, Ay, stride=6, title=None, arrow_scale=None):
    ax.imshow(phi.T, origin="lower", vmin=0.0, vmax=1.0,
              extent=[-L / 2, L / 2, -L / 2, L / 2])
    ax.contour(X, Y, phi, [0.5], colors="white", linewidths=0.8)
    if arrow_scale is None:
        arrow_scale = 14.0 * (np.hypot(Ax, Ay).max() + 1e-12)
    ax.quiver(X[::stride, ::stride], Y[::stride, ::stride],
              Ax[::stride, ::stride], Ay[::stride, ::stride],
              color="white", scale=arrow_scale, width=0.004)
    half = L / 2.0
    wall_start = wall_margin_frac * half
    ax.add_patch(plt.Rectangle((-wall_start, -wall_start), 2 * wall_start, 2 * wall_start,
                                fill=False, edgecolor="orange", lw=1.2, ls="--"))
    ax.set_xlim(-L / 2, L / 2)
    ax.set_ylim(-L / 2, L / 2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    if title:
        ax.set_title(title, fontsize=9)


# 2. The per-step physics, and the decision primitives

`advance_one_step` is steps A–E of Version 3's loop, taking the
treadmilling rate $w_0$ as a **field** rather than a scalar — the only
change needed so that, before the two cells merge, one blob's region
can be actively crawling while the other's is relaxed with no
treadmilling, using the same equations evaluated over the whole grid
at once. (`zeta_run` and `beta_run` already multiply other arrays
elementwise in `chemical_potential_phi`/`molecular_field_P`/`body_force`
above, so passing them as fields instead of scalars needs no changes
there at all — only `w0` needed the explicit field treatment.)


In [ ]:
def advance_one_step(phi, Px, Py, vx, vy, w0_field, dt=dt, flow=True):
    P2 = Px**2 + Py**2
    mu = chemical_potential_phi(phi, P2, d_dx(Px), d_dy(Py))
    hx, hy = molecular_field_P(phi, Px, Py, P2)

    if flow:
        fx, fy = body_force(phi, Px, Py, mu, hx, hy)
        vx, vy = stokes_solve(fx, fy)

    ux = vx + w0_field * Px
    uy = vy + w0_field * Py

    g = np.clip(phi * (1.0 - phi), 0.0, None)
    Lagrange = mu.sum() / g.sum()
    phi = phi + dt * (-divergence_of_flux(phi, ux, uy) - M_phi * (mu - Lagrange * g))

    dvx_dx, dvx_dy = d_dx(vx), d_dy(vx)
    dvy_dx, dvy_dy = d_dx(vy), d_dy(vy)
    Dxx, Dyy = dvx_dx, dvy_dy
    Dxy = 0.5 * (dvx_dy + dvy_dx)
    omega = 0.5 * (dvy_dx - dvx_dy)

    adv_Px = ux * d_dx(Px) + uy * d_dy(Px)
    adv_Py = ux * d_dx(Py) + uy * d_dy(Py)

    Px_new = Px + dt * (-adv_Px - omega * Py + chi_run * (Dxx * Px + Dxy * Py) - hx / Gamma)
    Py_new = Py + dt * (-adv_Py + omega * Px + chi_run * (Dxy * Px + Dyy * Py) - hy / Gamma)

    return phi, Px_new, Py_new, vx, vy


def decide_direction(rng, mean_theta=None, kappa=0.0):
    '''Uniform random heading, or a von-Mises draw biased around
    `mean_theta` with concentration `kappa` (higher = more biased).'''
    if mean_theta is None or kappa <= 0:
        return float(rng.uniform(0.0, 2.0 * np.pi))
    return float(rng.vonmises(mean_theta, kappa)) % (2.0 * np.pi)


def inward_theta(x, y):
    '''The direction pointing from (x, y) back towards the box centre --
    used to bias a redirect decision away from a wall.'''
    return float(np.arctan2(-y, -x))


def set_direction(phi, Px, Py, mask, theta):
    '''The 'decision': point P at theta, inside `mask` only.'''
    Px[mask] = phi[mask] * np.cos(theta)
    Py[mask] = phi[mask] * np.sin(theta)


# 3. Two cells, one box, one driver

Each cell's own state (current heading, phase, target distance, where
its current leg started, its last known velocity and mass) is tracked
in a small dict while the two blobs are still topologically separate
— identified every `sample_every` steps via connected-component
labelling (`scipy.ndimage.label`) on $\phi>0.5$, matched to the
previous check's two blobs by nearest centroid so identity doesn't
flicker between checks.

Three things can trigger a new decision for a given cell, checked in
this order: **(a)** its centroid has drifted inside the wall margin
— redirect, biased back towards the centre; **(b)** it is in its
*move* phase and has covered its target distance — switch to *relax*;
**(c)** it is in its *relax* phase and the fixed relax duration has
elapsed — pick a fresh random heading and switch back to *move*.

The moment `ndimage.label` reports **one** component instead of two,
the merge branch fires once: it computes $\mathbf p_{\rm tot}$ and
$L_{\rm tot}$ from the two cells' last known masses/velocities/
positions, reports them, and biases the merged cell's first new
heading towards $\mathbf p_{\rm tot}$'s direction. From then on there
is only one tracked state, following the same three triggers.


In [ ]:
def run_two_cell_walk(move=dict(w0=w0, zeta=zeta, beta=beta),
                      distance_range=(15.0, 35.0), relax_time=30.0,
                      wall_redirect_margin=15.0, wall_kappa=3.0, merge_kappa=2.0,
                      init_separation=50.0, t_final=400.0, dt=dt,
                      sample_every=sample_every, frame_every=300, seed=3,
                      verbose=True):
    """Two cells, each walking its own decide/move/relax cycle inside a
    walled box, merging into one on contact. Returns (phi, Px, Py,
    diagnostics, frames, merge_report)."""

    global zeta_run, beta_run, xi_run, chi_run, kappa_run
    xi_run, chi_run, kappa_run = xi, chi, kappa
    half = L / 2.0
    rng = np.random.default_rng(seed)

    # ---------------- two disjoint blobs, each with its own heading ----------------
    centers = [(-init_separation / 2.0, 0.0), (init_separation / 2.0, 0.0)]
    phi = np.zeros((N, N))
    Px = np.zeros((N, N))
    Py = np.zeros((N, N))
    for cx, cy in centers:
        r = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
        phi = np.maximum(phi, 0.5 * (1.0 - np.tanh((r - R0) / (np.sqrt(2.0) * eps))))

    states = []
    for cx, cy in centers:
        mask = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2) < 1.5 * R0
        theta = decide_direction(rng)
        set_direction(phi, Px, Py, mask, theta)
        states.append(dict(theta=theta, phase="move",
                            target=rng.uniform(*distance_range),
                            x0=cx, y0=cy, prev_xy=(cx, cy), relax_t0=0.0,
                            mass=None, prev_v=(0.0, 0.0)))

    vx, vy = np.zeros_like(phi), np.zeros_like(phi)
    t, step = 0.0, 0
    merged = False
    merge_report = None
    status_text = "starting"

    diagnostics = {"time": [], "cell": [], "x": [], "y": [], "phase": []}
    frames = []

    def record(cell_id, phase, x, y):
        diagnostics["time"].append(t)
        diagnostics["cell"].append(cell_id)
        diagnostics["x"].append(x)
        diagnostics["y"].append(y)
        diagnostics["phase"].append(phase)

    while t < t_final:
        labels, n_labels = ndimage.label(phi > 0.5)

        if not merged and n_labels >= 2:
            sizes = ndimage.sum(phi > 0.5, labels, index=range(1, n_labels + 1))
            top2 = (np.argsort(sizes)[-2:] + 1).tolist()
            cents = []
            for lbl in top2:
                mask = labels == lbl
                m = (phi * mask).sum()
                xc = (X * phi * mask).sum() / m
                yc = (Y * phi * mask).sum() / m
                cents.append([lbl, xc, yc, m, mask])

            d_direct = (abs(cents[0][1] - states[0]['prev_xy'][0])
                        + abs(cents[1][1] - states[1]['prev_xy'][0]))
            d_swap = (abs(cents[1][1] - states[0]['prev_xy'][0])
                      + abs(cents[0][1] - states[1]['prev_xy'][0]))
            if d_swap < d_direct:
                cents = [cents[1], cents[0]]

            zeta_field = np.zeros_like(phi)
            beta_field = np.zeros_like(phi)
            w0_field = np.zeros_like(phi)

            for i, (lbl, xc, yc, m, mask) in enumerate(cents):
                s = states[i]
                vxi = (xc - s['prev_xy'][0]) / (sample_every * dt)
                vyi = (yc - s['prev_xy'][1]) / (sample_every * dt)
                s['prev_v'] = (vxi, vyi)
                s['mass'] = m
                s['prev_xy'] = (xc, yc)

                travelled = np.hypot(xc - s['x0'], yc - s['y0'])
                edge_dist = half - max(abs(xc), abs(yc))
                record(i, s['phase'], xc, yc)

                if edge_dist < wall_redirect_margin:
                    theta = decide_direction(rng, inward_theta(xc, yc), wall_kappa)
                    s.update(theta=theta, phase="move",
                             target=rng.uniform(*distance_range), x0=xc, y0=yc)
                    set_direction(phi, Px, Py, mask, theta)
                    if verbose:
                        print(f"t={t:6.1f}  cell {i}: near the wall, "
                              f"redirect to {np.degrees(theta):.0f} deg")
                elif s['phase'] == 'move' and travelled >= s['target']:
                    s.update(phase='relax', relax_t0=t)
                    if verbose:
                        print(f"t={t:6.1f}  cell {i}: reached target "
                              f"({travelled:.1f}), relaxing")
                elif s['phase'] == 'relax' and (t - s['relax_t0']) >= relax_time:
                    theta = decide_direction(rng)
                    s.update(theta=theta, phase='move',
                             target=rng.uniform(*distance_range), x0=xc, y0=yc)
                    set_direction(phi, Px, Py, mask, theta)
                    if verbose:
                        print(f"t={t:6.1f}  cell {i}: relaxed, new direction "
                              f"{np.degrees(theta):.0f} deg")

                if states[i]['phase'] == 'move':
                    zeta_field[mask] = move['zeta']
                    beta_field[mask] = move['beta']
                    w0_field[mask] = move['w0']

        elif not merged and n_labels == 1:
            # ---------------- MERGE EVENT ----------------
            merged = True
            m0 = states[0]['mass'] if states[0]['mass'] is not None else 1.0
            m1 = states[1]['mass'] if states[1]['mass'] is not None else 1.0
            v0, v1 = states[0]['prev_v'], states[1]['prev_v']
            r0, r1 = states[0]['prev_xy'], states[1]['prev_xy']
            p_total = (m0 * v0[0] + m1 * v1[0], m0 * v0[1] + m1 * v1[1])
            Rcm = ((m0 * r0[0] + m1 * r1[0]) / (m0 + m1),
                   (m0 * r0[1] + m1 * r1[1]) / (m0 + m1))
            L0 = m0 * ((r0[0] - Rcm[0]) * v0[1] - (r0[1] - Rcm[1]) * v0[0])
            L1 = m1 * ((r1[0] - Rcm[0]) * v1[1] - (r1[1] - Rcm[1]) * v1[0])
            L_total = L0 + L1
            p_mag = np.hypot(*p_total)
            mean_theta = np.arctan2(p_total[1], p_total[0]) if p_mag > 1e-6 else None
            theta = decide_direction(rng, mean_theta, merge_kappa if p_mag > 1e-6 else 0.0)

            xc, yc = centre_of_mass(phi)
            full_mask = phi > 0.0
            set_direction(phi, Px, Py, full_mask, theta)
            states = [dict(theta=theta, phase='move',
                            target=rng.uniform(*distance_range), x0=xc, y0=yc,
                            prev_xy=(xc, yc), relax_t0=0.0, mass=phi.sum(), prev_v=(0.0, 0.0))]
            merge_report = dict(t=t, p_total=p_total, L_total=L_total,
                                 Rcm=Rcm, new_theta_deg=float(np.degrees(theta)))
            record('merged', 'move', xc, yc)
            if verbose:
                print(f"\nt={t:6.1f}  *** MERGE ***   "
                      f"p_total = ({p_total[0]:+.3f}, {p_total[1]:+.3f})   "
                      f"L_total = {L_total:+.4f}   "
                      f"-> new direction {np.degrees(theta):.0f} deg\n")

            zeta_field = move['zeta'] * full_mask
            beta_field = move['beta'] * full_mask
            w0_field = move['w0'] * full_mask

        else:  # already merged
            s = states[0]
            xc, yc = centre_of_mass(phi)
            s['prev_xy'] = (xc, yc)
            travelled = np.hypot(xc - s['x0'], yc - s['y0'])
            edge_dist = half - max(abs(xc), abs(yc))
            record('merged', s['phase'], xc, yc)
            full_mask = phi > 0.0

            if edge_dist < wall_redirect_margin:
                theta = decide_direction(rng, inward_theta(xc, yc), wall_kappa)
                s.update(theta=theta, phase='move',
                         target=rng.uniform(*distance_range), x0=xc, y0=yc)
                set_direction(phi, Px, Py, full_mask, theta)
            elif s['phase'] == 'move' and travelled >= s['target']:
                s.update(phase='relax', relax_t0=t)
            elif s['phase'] == 'relax' and (t - s['relax_t0']) >= relax_time:
                theta = decide_direction(rng)
                s.update(theta=theta, phase='move',
                         target=rng.uniform(*distance_range), x0=xc, y0=yc)
                set_direction(phi, Px, Py, full_mask, theta)

            if s['phase'] == 'move':
                zeta_field = move['zeta'] * full_mask
                beta_field = move['beta'] * full_mask
                w0_field = move['w0'] * full_mask
            else:
                zeta_field = np.zeros_like(phi)
                beta_field = np.zeros_like(phi)
                w0_field = np.zeros_like(phi)

        zeta_run, beta_run = zeta_field, beta_field

        for _ in range(sample_every):
            phi, Px, Py, vx, vy = advance_one_step(phi, Px, Py, vx, vy, w0_field, dt=dt)
            t += dt
            step += 1

            if step % frame_every == 0:
                if merged:
                    status_text = f"t={t:5.1f}  MERGED  {states[0]['phase'].upper()}"
                    if states[0]['phase'] == 'move':
                        status_text += f"  dir={np.degrees(states[0]['theta']):.0f} deg"
                else:
                    parts = []
                    for i, s in enumerate(states):
                        tag = s['phase'].upper()
                        if s['phase'] == 'move':
                            tag += f" {np.degrees(s['theta']):.0f} deg"
                        parts.append(f"cell{i}: {tag}")
                    status_text = f"t={t:5.1f}   " + "   ".join(parts)
                frames.append((t, phi.copy(), Px.copy(), Py.copy(), status_text))

            if touches_boundary(phi):
                print(f"   ** WARNING: phi reached the literal array edge at "
                      f"t={t:.1f}; raise k_wall or widen the wall margin")

            if not (np.isfinite(phi).all() and np.isfinite(Px).all() and np.isfinite(Py).all()):
                print(f"** diverged at t = {t:.2f}")
                diagnostics = {k: (v if k in ("cell", "phase") else np.array(v))
                               for k, v in diagnostics.items()}
                return phi, Px, Py, diagnostics, frames, merge_report

            if t >= t_final:
                break

    diagnostics = {k: (v if k in ("cell", "phase") else np.array(v)) for k, v in diagnostics.items()}
    return phi, Px, Py, diagnostics, frames, merge_report


# 4. Run it

**Not executed here** — run it on your own machine. $N=192$ (versus
$128$ for one cell) costs roughly $2$–$2.5\times$ per step from the
larger FFTs; at $t_{\rm final}=400$ that is on the order of
$40\,000$ steps. Expect somewhere in the neighbourhood of $4$–$5$
minutes, as a reasoned estimate rather than a measured one. Shorten
`t_final` to iterate faster while you check that `k_wall` is actually
stopping the cells at the wall (watch for the boundary warning).


In [ ]:
phi_f, Px_f, Py_f, diag_tc, frames_tc, merge_info = run_two_cell_walk(
    move=dict(w0=1.0, zeta=-1.0, beta=0.25),
    distance_range=(15.0, 35.0), relax_time=30.0,
    wall_redirect_margin=15.0, wall_kappa=3.0, merge_kappa=2.0,
    init_separation=50.0, t_final=400.0, frame_every=300, seed=3)

print(f"\n{len(frames_tc)} frames captured over t = 0..{diag_tc['time'][-1]:.0f}")
if merge_info:
    print(f"merge at t={merge_info['t']:.1f}, "
          f"p_total={merge_info['p_total']}, L_total={merge_info['L_total']:+.4f}, "
          f"new heading {merge_info['new_theta_deg']:.0f} deg")
else:
    print("the two cells never met during this run -- try a smaller "
          "init_separation, a longer t_final, or a different seed")


# 5. Diagnostics — two paths becoming one

In [ ]:
cell_arr = np.array(diag_tc["cell"], dtype=object)
phase_arr = np.array(diag_tc["phase"], dtype=object)

fig, ax = plt.subplots(figsize=(7, 7))
colors = {0: "tab:blue", 1: "tab:red", "merged": "tab:green"}
for cid in (0, 1, "merged"):
    m = cell_arr == cid
    if not m.any():
        continue
    move_m = m & (phase_arr == "move")
    relax_m = m & (phase_arr == "relax")
    ax.plot(diag_tc["x"][move_m], diag_tc["y"][move_m], ".", color=colors[cid],
            ms=4, label=f"{cid} (move)" if cid != "merged" else "merged (move)")
    ax.plot(diag_tc["x"][relax_m], diag_tc["y"][relax_m], ".", color=colors[cid],
            ms=4, alpha=0.35)

if merge_info is not None:
    ax.plot(*merge_info["Rcm"], "k*", ms=18, label="merge point")

half = L / 2.0
wall_start = wall_margin_frac * half
ax.add_patch(plt.Rectangle((-half, -half), L, L, fill=False, edgecolor="black", lw=1))
ax.add_patch(plt.Rectangle((-wall_start, -wall_start), 2 * wall_start, 2 * wall_start,
                            fill=False, edgecolor="orange", lw=1.2, ls="--"))
ax.set_aspect("equal")
ax.set(xlabel="x", ylabel="y",
       title="Two independent walks, merging on contact\n"
             "(orange dashed = where the soft wall turns on)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("two_cell_walk_path.png", dpi=140)
plt.show()


# 6. A video of the whole walk, collision, and merge

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 6.6))


def render(i):
    ax.clear()
    t, phi_i, Px_i, Py_i, status = frames_tc[i]
    draw_cell(ax, phi_i, Px_i, Py_i, title=status)


render(0)
fig.tight_layout()


def update(i):
    render(i)
    return []


ani = animation.FuncAnimation(fig, update, frames=len(frames_tc), blit=False)

fps = 10
video_path = "two_cell_walk.mp4"
ani.save(video_path, writer=animation.FFMpegWriter(fps=fps, bitrate=2800))
plt.close(fig)

print(f"saved {video_path}: {len(frames_tc)} frames at {fps} fps "
      f"= {len(frames_tc) / fps:.1f} s")


In [ ]:
Video(video_path, embed=True, width=480)


# 7. Notes, and open questions I'd want your call on

- **The wall is a potential, not a boundary condition.** $k_{\rm
  wall}=40$ and the $80\%$ margin are reasoned starting points, not
  something I ran and tuned. If a cell punches through, raise
  $k_{\rm wall}$ or widen the margin first; the warning in Section 3
  will tell you if it happens.
- **"Angular momentum" here is a bias on a random choice, not a
  conserved quantity.** Said plainly in Section 0 and worth repeating:
  Stokes flow has no inertia to carry a collision's spin forward, so
  $L_{\rm tot}$ is reported and used to bias the post-merge heading's
  *distribution*, not enforced as a dynamical law the equations
  don't have.
- **Relax duration is a fixed time here**, not the circularity-based
  criterion the single-cell notebook used — simpler, and avoids
  computing a shape metric for a connected component whose identity
  can flicker between checks while two blobs are both moving.
- **Open questions for you:**
  - Is a fixed `relax_time` (currently $30$) the right call, or
    should relaxation also check actual circularity, accepting the
    added bookkeeping?
  - Right now `merge_kappa` biases the *heading*; should the merge
    also seed a small swirl in $\mathbf P$ proportional to $L_{\rm
    tot}$'s sign, so the merged cell visibly starts to curve one way
    rather than only picking a biased straight heading? Easy to add —
    I didn't, to keep the first version legible.
  - Extending past two cells needs little beyond generalising the
    `states` list and the `top2`/`cents` matching logic to $N$ blobs
    — worth doing once this two-cell version is checked against a
    real run.
  - The honest fix for the wall is a real non-periodic Stokes solve
    (finite-difference Poisson/Helmholtz with wall boundary
    conditions, replacing the two-line spectral solve with a sparse
    linear solve). Bigger project; flagging it rather than doing it
    quietly via the soft potential forever.
